In [ ]:
%load_ext autoreload
%autoreload 2
import jax.numpy as jnp
from popsim.scenarios.sparc_prd.comet_mirror import build_comet_mirror_config
from popsim.simulators.comet_mirror.simulate import simulate
import jax
from popsim.config_utils import MultiCases, build_configs
from popsim.gui import PopsimGUI
import panel as pn
import dataclasses

# Initialize the simulator.
model, state, params = build_comet_mirror_config()
ts = jnp.linspace(0, 2, 100)

# Generate Random Walks for Particle Confinement Times

In [ ]:
from popsim.stochastic import generate_random_walks

diffusion_mags = {k: 0.15 for k in params.particle_confinement_scalar.keys()}
n_samps = 10
sols = generate_random_walks(
    jax.random.PRNGKey(42),
    n_samps,
    ts,
    params.particle_confinement_scalar,
    diffusion_mags,
)
# k_particle_trajs = {k: cubic_interp(sols.ts[0], sols.ys[k].T) for k in sols.ys}
# visualize_time_series(solution_to_xarray(sols, True)).cols(2)

# Build a list of parameters, one for each random walk trajectory.

In [ ]:
params = dataclasses.replace(params, particle_confinement_scalar=MultiCases(cases=sols))
dataset = simulate(model, ts, state, params, return_xarray=True)

# Visualize the simulation state results along with some debug variables

In [ ]:
# gui_hook = PopsimGUI(
#     dataset,
#     time_dim="time",
#     rho_dim="rho",
#     simulation_dim="simulation",
# )
# pn.serve(gui_hook.build_view(), port=8080)